# Render EEVEE (toon) di Colab GPU

1. **Runtime → Change runtime type → T4 GPU** (atau L4/A100).
2. Isi cell **Config**, lalu *Run all*.
3. Kalau sesi putus: sambung lagi, *Run all* lagi. Frame yang sudah jadi (disimpan di Google Drive) **di-skip**, render lanjut dari yang belum.

In [ ]:
#@title Config
REPO_URL   = "https://github.com/USERNAME/shorts3d.git"  #@param {type:"string"}
BRANCH     = "main"                                       #@param {type:"string"}
GH_TOKEN   = ""   #@param {type:"string"}  isi kalau repo private (Personal Access Token, read-only)
SLUG       = "test_e2e"                                   #@param {type:"string"}
BUILD_SCRIPT = "test_e2e/scene.py"  #@param {type:"string"}  script yang membangun .blend (anim.py untuk video)
RES_PCT    = 100   #@param {type:"integer"}  100 = 1080x1920, 50 = preview 540x960
SAMPLES    = 16    #@param {type:"integer"}
CHUNK      = 30    #@param {type:"integer"}  frame per panggilan blender (progress & checkpoint)
AUDIO      = ""    #@param {type:"string"}  path wav final relatif repo (kosong = video bisu)
BLENDER_VER = "4.2.3"
USE_DRIVE  = True  #@param {type:"boolean"}

In [ ]:
#@title 1. GPU + Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/shorts3d_render'
else:
    WORK = '/content/shorts3d_render'
os.makedirs(WORK, exist_ok=True)
print('WORK =', WORK)

In [ ]:
#@title 2. Install dependency + EGL NVIDIA (biar EEVEE pakai GPU, bukan software)
!apt-get -qq update > /dev/null
!apt-get -qq install -y libegl1 libgl1 libglvnd0 libxi6 libxkbcommon0 libxrender1 libsm6 libxxf86vm1 libxfixes3 ffmpeg > /dev/null
import os, glob, json
# Colab memasang driver NVIDIA di /usr/lib64-nvidia; daftarkan vendor EGL-nya ke glvnd
nv = glob.glob('/usr/lib64-nvidia/libEGL_nvidia.so*') + glob.glob('/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so*')
print('libEGL_nvidia:', nv)
os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as f:
    json.dump({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}, f)
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'

In [ ]:
#@title 3. Install Blender portable (tarball di-cache di Drive)
import os
mm = '.'.join(BLENDER_VER.split('.')[:2])
tar = f'{WORK}/blender-{BLENDER_VER}-linux-x64.tar.xz'
if not os.path.exists(tar):
    !wget -q -O "{tar}" https://download.blender.org/release/Blender{mm}/blender-{BLENDER_VER}-linux-x64.tar.xz
if not os.path.exists(f'/content/blender-{BLENDER_VER}-linux-x64'):
    !tar xf "{tar}" -C /content
BLENDER = f'/content/blender-{BLENDER_VER}-linux-x64/blender'
!{BLENDER} --version | head -1

In [ ]:
#@title 4. Clone / update repo
import os
url = REPO_URL.replace('https://', f'https://{GH_TOKEN}@') if GH_TOKEN else REPO_URL
if os.path.exists('/content/repo/.git'):
    !cd /content/repo && git fetch -q origin {BRANCH} && git reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} "{url}" /content/repo
!cd /content/repo && git log --oneline -1

In [ ]:
#@title 5. Cek GPU dipakai EEVEE (harus NVIDIA/Tesla, BUKAN llvmpipe)
out = !{BLENDER} -b --factory-startup -P /content/repo/tools/gpu_info.py 2>&1 | grep "\[gpu\]"
print('\n'.join(out))
if any('llvmpipe' in l for l in out) or not out:
    print('\n!! EEVEE TIDAK pakai GPU. Cek Runtime = GPU, lalu restart runtime & Run all.')

In [ ]:
#@title 6. Build scene (.blend) dari script
BLEND = f'{WORK}/{SLUG}/scene.blend'
os.makedirs(f'{WORK}/{SLUG}', exist_ok=True)
!cd /content/repo && {BLENDER} -b -P {BUILD_SCRIPT} -- --out "{BLEND}" 2>&1 | grep -E "saved|Error|Traceback" 

In [ ]:
#@title 7. Render per chunk (RESUMABLE) + laporan waktu per frame
import subprocess, time, json, glob, os, statistics
FR = f'{WORK}/{SLUG}/frames_{RES_PCT}'
os.makedirs(FR, exist_ok=True)
info = subprocess.run([BLENDER, '-b', BLEND, '--python-expr',
    'import bpy;s=bpy.context.scene;print("RANGE",s.frame_start,s.frame_end)'], capture_output=True, text=True).stdout
fs, fe = [int(x) for x in [l for l in info.splitlines() if l.startswith('RANGE')][0].split()[1:]]
todo = [f for f in range(fs, fe + 1) if not os.path.exists(f'{FR}/{f:05d}.png')]
print(f'frame {fs}-{fe}: {fe-fs+1-len(todo)} sudah ada, {len(todo)} sisa')
t0 = time.time()
for a in range(fs, fe + 1, CHUNK):
    b = min(a + CHUNK - 1, fe)
    if all(os.path.exists(f'{FR}/{f:05d}.png') for f in range(a, b + 1)):
        continue
    subprocess.run([BLENDER, '-b', BLEND, '-P', '/content/repo/tools/render_frames.py', '--',
                    '--out', FR, '--start', str(a), '--end', str(b),
                    '--pct', str(RES_PCT), '--samples', str(SAMPLES)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    done = len(glob.glob(f'{FR}/*.png'))
    print(f'  chunk {a}-{b} ok | {done}/{fe-fs+1} frame | {time.time()-t0:.0f}s')
secs = [json.loads(l)['sec'] for l in open(f'{FR}/timing.jsonl')]
if secs:
    print(f'\nwaktu/frame: median {statistics.median(secs):.2f}s, min {min(secs):.2f}s, max {max(secs):.2f}s ({len(secs)} frame)')

In [ ]:
#@title 8. Encode mp4 (+ audio kalau ada) & download
os.makedirs(f'{WORK}/{SLUG}/render', exist_ok=True)
MP4 = f'{WORK}/{SLUG}/render/{SLUG}_{RES_PCT}.mp4'
aud = f'/content/repo/{AUDIO}' if AUDIO else ''
!bash /content/repo/tools/encode.sh "{FR}" "{MP4}" {aud}
!ffprobe -v error -show_entries format=duration:stream=width,height,r_frame_rate -of compact "{MP4}"
from google.colab import files
files.download(MP4)